# Gold Layer — agg_restaurant_comments (added 2026-11-01)

**Grain:** 1 restaurant (`restaurant_id`) × 1 comment on a video that mentions it. **Full overwrite** daily (`CREATE OR REPLACE`). Run after `gold/checks`, before `gold/agg_restaurant_ranking` (which sums it into `comment_count` / `positive_comment_share` / `negative_comment_share`). Read by the dashboard page "Khán giả nói gì".

- A comment under a "top 10 quán" video is not tied to one restaurant, so it counts for every restaurant of the video with `mention_weight` (1 / restaurants in the video), like views.
- `sentiment` / `about_food` come from `silver.stg_comment_sentiment` (NULL until analyzed).
- Only videos with `dim_video.is_food_review` (classifier) count.

In [ ]:
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))
TARGET = f"{GOLD}.agg_restaurant_comments"
SENTIMENT = f"{SILVER}.stg_comment_sentiment"
dv_cols = set(spark.table(f"{GOLD}.dim_video").columns)
food_filter = "COALESCE(v.is_food_review, true)" if "is_food_review" in dv_cols else "true"
sentiment_join = (f"LEFT JOIN {SENTIMENT} s ON s.comment_id = f.comment_id" if spark.catalog.tableExists(SENTIMENT)
                  else "LEFT JOIN (SELECT CAST(NULL AS STRING) AS comment_id, CAST(NULL AS STRING) AS sentiment, "
                       "CAST(NULL AS BOOLEAN) AS about_food WHERE false) s ON s.comment_id = f.comment_id")
print(f"Gold target: {TARGET}")

In [ ]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET}
    COMMENT 'Restaurant x comment on a video that mentions it (split credit by mention_weight), with LLM sentiment. Full overwrite daily.'
    AS
    SELECT
        d.restaurant_id,
        f.comment_id,
        v.video_id,
        v.title                    AS video_title,
        v.video_url,
        f.comment_text,
        f.like_count,
        f.reply_count,
        f.posted_at,
        m.mention_weight,
        s.sentiment,
        s.about_food,
        current_timestamp()        AS etl_date
    FROM {GOLD}.fact_comment f
    JOIN {GOLD}.dim_video v ON v.video_key = f.video_key AND f.video_key <> -1
    JOIN {GOLD}.fact_video_restaurant_mention m ON m.video_key = f.video_key AND m.restaurant_key <> -1
    JOIN {GOLD}.dim_restaurant d ON d.restaurant_key = m.restaurant_key
    {sentiment_join}
    WHERE {food_filter}
""")
s = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(DISTINCT restaurant_id, comment_id) AS grain, COUNT(DISTINCT restaurant_id) AS restaurants,
           COUNT(sentiment) AS analyzed
    FROM {TARGET}
""").collect()[0]
print(f"📊 {TARGET}: {s['n']} rows, {s['restaurants']} restaurants, {s['analyzed']} with sentiment "
      + ("✅" if s["n"] == s["grain"] else "❌ duplicate grain"))
assert s["n"] == s["grain"], "duplicate restaurant x comment rows"